In [1]:
%matplotlib widget
import torch

from thesis_code.datasets import get_SHD_dataset
from thesis_code.networks import models
from thesis_code.plotting import plot_performance, plot_gradients, InteractiveSpikePlot

In [2]:
torch.manual_seed(42)
dtype = torch.float
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"Using device: {device}")

Using device: cpu


Timesteps are in milliseconds

In [3]:
dt = 0.001

In [4]:
batch_size = 128
SHD_trainloader, SHD_testloader, input_dim, classes = get_SHD_dataset(batch_size, device)

In [ ]:
import tonic
from tonic import DiskCachedDataset
import tonic.transforms as transforms
from torch.utils.data import DataLoader
import shutil
from pathlib import Path

PROJECT_ROOT = Path("C:\\Users\\emiel\\Documents\\Study\\Thesis\\Thesis-Code")
DATASETS_DIR = PROJECT_ROOT / "datasets"

def get_SHD_dataloader(batch_size: int, time_window: float = 1000, train: bool = True, shuffle: bool = True, drop_last: bool = False, re_download: bool = False) -> tuple[DataLoader, int, int]:
    train_extension = "train" if train else "test"
    cache_path = DATASETS_DIR / "cache" / "SHD" / train_extension
    data_path = DATASETS_DIR / "data"

    if re_download:
        shutil.rmtree(cache_path, ignore_errors=True)
        shutil.rmtree(DATASETS_DIR / "data" / "SHD", ignore_errors=True)

    sensor_size = tonic.datasets.SHD.sensor_size
    frame_transform=transforms.ToFrame(
        sensor_size=sensor_size,
        time_window=time_window,
        start_time=0,
    )
    dataset = tonic.datasets.SHD(
        save_to=str(data_path), 
        train=train,
        transform=frame_transform,
    )
    cached_dataset = DiskCachedDataset(
        dataset,
        cache_path=str(cache_path)
    )
    dataloader = DataLoader(
        cached_dataset,
        batch_size=batch_size,
        collate_fn=tonic.collation.PadTensors(batch_first=False),
        shuffle=shuffle,
        drop_last=drop_last,
    )
    return dataloader, sensor_size[0], dataset.classes


SHD_trainloader, input_dim, classes = get_SHD_dataloader(
    batch_size=128,
    time_window=1000,
    train=True,
    shuffle=True,
    drop_last=True,
    re_download=False,
)

SHD_testloader, _, _ = get_SHD_dataloader(
    batch_size=128,
    time_window=1000,
    train=False,
    shuffle=True,
    drop_last=True,
    re_download=False,
)

  0%|          | 0/130863613 [00:00<?, ?it/s]

Extracting C:\Users\emiel\Documents\Study\Thesis\Thesis-Code\datasets\data\SHD\shd_train.h5.zip to C:\Users\emiel\Documents\Study\Thesis\Thesis-Code\datasets\data\SHD


  0%|          | 0/38141465 [00:00<?, ?it/s]

Extracting C:\Users\emiel\Documents\Study\Thesis\Thesis-Code\datasets\data\SHD\shd_test.h5.zip to C:\Users\emiel\Documents\Study\Thesis\Thesis-Code\datasets\data\SHD


In [10]:
import torch.nn as nn
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
import numpy as np
import torch.nn.functional as F
from snntorch import surrogate
from typing import Callable

from thesis_code.training import get_compute_loss_reg_fn
from thesis_code.decay_sampling import sample_heterogeneous_decays_normal, tau_to_beta

def compute_loss(mem_rec: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
    probs = F.softmax(mem_rec, dim=-1)
    log_probs = torch.log(probs.mean(dim=0) + 1e-8) # to avoid log(0)
    return F.nll_loss(log_probs, targets)

def measure_accuracy(mem_rec, targets):
    _, idx = mem_rec.sum(dim=0).max(1)
    return np.mean((targets == idx).detach().cpu().numpy())

def test(net: nn.Module, device: torch.device, testloader: DataLoader, max_iters: int = None) -> float:
    with torch.no_grad():
        acc = 0
        net.eval()

        print(len(testloader))

        print("Her")
        for i, (data, targets) in enumerate(testloader):
            data, targets = data.to(device), targets.to(device).long()
            data = data.squeeze(2)
            print(i)
            
            spk_rec, mem_rec, hidden_spks = net(data)
            acc += measure_accuracy(mem_rec, targets)
            if max_iters is not None and i == max_iters:
                break
            print("End")
    return acc / len(testloader)

def record_grads(net: nn.Module) -> tuple[list, list]:
    grads = {name: p.grad.detach().clone()
        for name, p in net.named_parameters() if p.grad is not None}
    if len(grads) > 0:
        flat_grads = torch.cat([t.flatten() for t in grads.values()])
        return flat_grads.max(dim=0).values.cpu().numpy(), flat_grads.mean(dim=0).cpu().numpy()
    return 0, 0

def train(net: nn.Module, device: torch.device, trainloader: DataLoader, loss_fn: Callable, lr: float = 1e-3, n_epochs: int = 1, max_iters: int = None) -> tuple[nn.Module, list, list]:
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, eps=1e-6, betas=(0.9, 0.99))

    loss_hist, acc_hist, spike_hist = [], [], []
    net.train()
    max_grad_rec = []
    avg_grad_rec = []

    for epoch in range(n_epochs):
        progress_bar = tqdm(enumerate(trainloader), total=len(trainloader), desc=f"Epoch {epoch}")
        for i, (data, targets) in progress_bar:
            data, targets = data.to(device), targets.to(device).long()
            data = data.squeeze(2)

            spk_outs, mem_outs, hidden_spks = net(data)
            loss_val = loss_fn(mem_outs, targets, hidden_spks)

            optimizer.zero_grad()
            loss_val.backward()

            # torch.nn.utils.clip_grad_value_(net.parameters(), clip_value=1e6)

            max_grad, avg_grad = record_grads(net)
            max_grad_rec.append(max_grad)
            avg_grad_rec.append(avg_grad)

            optimizer.step()

            loss_hist.append(loss_val.item())

            acc = measure_accuracy(mem_outs, targets)
            acc_hist.append(acc)

            progress_bar.set_postfix(loss=f"{loss_val.item():.2f}", acc=f"{acc * 100:.2f}%")

            if i % 25 == 0:
                tqdm.write(f"Epoch {epoch}, Iteration {i} — Loss: {loss_val.item():.2f}, Acc: {acc*100:.2f}%")

            if max_iters is not None and i == max_iters:
                break

    return net, loss_hist, acc_hist, spike_hist, max_grad_rec, avg_grad_rec

In [11]:
from thesis_code.connectivity import sample_ee_pv_som_neurons, build_recurrent_matrices

ns_hidden=[256, 256]
neurons = sample_ee_pv_som_neurons(ns_hidden, percent_pv=25.0, percent_som=25.0)
rec_matrices_pv_som_ee = build_recurrent_matrices(neurons)

In [12]:
from snntorch.surrogate import fast_sigmoid

from thesis_code.network_components.layers import OutputLayer, StandardRecurrentLayer
from thesis_code.recording import NeuronRecorder

class TestSRNN(nn.Module):
    def __init__(self,
        forward_matrices: list[torch.Tensor],
        recurrent_matrices: list[torch.Tensor],
        betas: list[torch.Tensor | float],
        beta_out: float,
        n_classes: int,
        spike_grad = fast_sigmoid(slope=25),
        fully_learnable: bool = True,
        record: bool = True,
    ) -> None:
        super().__init__()

        self.rec_layers = nn.ModuleList()

        assert len(forward_matrices) == len(recurrent_matrices) == len(betas), \
            "unequal number forward- and recurrent matrices and betas"

        for fm_prev, fm_next in zip(forward_matrices, forward_matrices[1:]):
            assert fm_prev.shape[1] == fm_next.shape[0], \
                f"layer output {fm_prev.shape[1]} doesn't match next layer input {fm_next.shape[0]}"

        for fm, rm, beta in zip(forward_matrices, recurrent_matrices, betas):
            assert fm.shape[-1] == rm.shape[0], \
                f"forward matrix outputs {fm.shape[-1]} neurons, recurrent matrix has {rm.shape[0]}"

            self.rec_layers.append(StandardRecurrentLayer(
                forward_matrix=fm,
                recurrent_matrix=rm,
                beta=beta,
                spike_grad=spike_grad,
                fully_learnable=fully_learnable,
                record=record,
            ))

        self.out_layer = OutputLayer(forward_matrices[-1].shape[-1], n_classes, beta_out)
        self.recorder = NeuronRecorder(num_layers=len(recurrent_matrices))
        self.record = record

    def reset(self) -> None:
        for layer in self.rec_layers:
            layer.reset()
        self.out_layer.reset()

    def record_layers(self) -> None:
        for i, layer in enumerate(self.rec_layers):
            self.recorder.add_layer_recordings(i, *layer.get_recordings())
        self.recorder.increment_iterations()

    def get_recorder(self) -> NeuronRecorder:
        assert self.record, "recording is disabled (record=False)"
        return self.recorder

    def forward(self, data: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor, list[torch.Tensor]]:
        self.reset()

        # with nn.utils.parametrize.cached():
        # spk_outs, mem_outs, hidden_spks = self.forward_net(data)

        spk_outs, mem_outs = [], []
        hidden_spks = [[] for _ in range(len(self.rec_layers))]

        print("Start")

        for step in range(data.size(0)):
            # print("Continue")

            x = data[step]
            for i, layer in enumerate(self.rec_layers):
                x = layer(x)
                hidden_spks[i].append(x)

            spk_out, mem_out = self.out_layer(x)
            spk_outs.append(spk_out)
            mem_outs.append(mem_out)

        print("End1")

        if self.record:
            self.record_layers()

        return torch.stack(spk_outs, dim=0), torch.stack(mem_outs, dim=0), [torch.stack(spks_h, dim=0) for spks_h in hidden_spks]


In [13]:
layers = [input_dim] + ns_hidden

net_pv_som_ee = TestSRNN(
    forward_matrices=[torch.ones(n_in, n_out) for n_in, n_out in zip(layers[:-1], layers[1:])],
    recurrent_matrices=[torch.ones(n, n) for n in ns_hidden], # rec_matrices_pv_som_ee,
    betas=[sample_heterogeneous_decays_normal(dt, n, tau_mean=10 * dt, tau_std=1 * dt, device=device) for n in ns_hidden],
    beta_out=tau_to_beta(dt, 20 * dt),
    n_classes=len(classes),
    spike_grad=surrogate.fast_sigmoid(slope=25),
).to(device)

compute_loss_reg = get_compute_loss_reg_fn(
    lam_lower=100.0,
    v_lower=1e-2,               # min n spikes
    lam_uppers=[0.5, 0.5],
    v_uppers=[50, 50],          # max n spikes per layer
    L=2,
)

print("Started training")

accuracy = test(
    net=net_pv_som_ee,
    device=device,
    testloader=SHD_testloader,
    max_iters=None,
)
print(f"Test Accuracy before training is: {accuracy}")

net_pv_som_ee, loss_hist, acc_hist, spike_hist, max_grad_rec, avg_grad_rec = train(
    net=net_pv_som_ee,
    device=device,
    trainloader=SHD_trainloader,
    loss_fn=lambda mem_outs, targets, hidden_spks: compute_loss(mem_outs, targets) + compute_loss_reg(hidden_spks),
    lr=1e-3,
    n_epochs=50,
    max_iters=None,
)
plot_performance(loss_hist, acc_hist)
# plot_gradients(max_grad_rec, avg_grad_rec)

accuracy = test(
    net=net_pv_som_ee,
    device=device,
    testloader=SHD_testloader,
    max_iters=None,
)
print(f"Test Accuracy after training is: {accuracy}")

Started training
17
Her
0
Start
End1
End
1
Start


KeyboardInterrupt: 